# Prompt/schema experiment runner

The runner reads the registry created by `01_colab_experiment_setup.ipynb`, extracts trial KGs, and records automatic metrics. It is intended for TRAIN/DEV only. TEST is deliberately blocked so that prompts and schemas cannot be selected after inspecting held-out results.

Supported modes:

- `prompts`: vary prompts while keeping the relation count fixed;
- `relations`: vary the relation count while keeping the prompt fixed;
- `factorial`: run every prompt × relation-count combination;
- `baseline`: run one baseline configuration, useful for comparing only dataset sizes;
- `custom`: apply explicit user-defined filters.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Select the experiment

First leave `EXECUTE_RUNS = False` and inspect the printed plan. Then change it to `True`.

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/DKR-KG-Construction/KG-construction.git'
REPO_BRANCH = 'main'
REPO_DIR = Path('/content/KG-construction')
DRIVE_ROOT = Path('/content/drive/MyDrive/DKR_KG_Construction')
EXPERIMENT_ROOT = DRIVE_ROOT / 'configuration_search_full96'
REGISTRY_PATH = EXPERIMENT_ROOT / 'experiment_registry.json'

SPLIT = 'train'                 # train | dev; test is blocked
FOCUS = 'prompts'              # prompts | relations | factorial | baseline | custom
SIZE_MODE = 'small'            # pilot | small | medium | large | all
EXECUTE_RUNS = False           # inspect the plan first, then change to True
EXPORT_TRIAL_KG = True
KG_CONFIGS = ['raw', 'C', 'C+S', 'C+S+P']

# Used only when FOCUS='custom'. An empty list means 'do not filter'.
CUSTOM_PROMPTS = []            # for example ['baseline', 'conservative']
CUSTOM_RELATION_COUNTS = []     # for example [10, 14]

if SPLIT not in {'train', 'dev'}:
    raise ValueError('Configuration search is allowed only on TRAIN/DEV, never TEST')
print({'split': SPLIT, 'focus': FOCUS, 'size_mode': SIZE_MODE, 'execute': EXECUTE_RUNS})

## 2. Repository and dependencies

In [ ]:
import json
import os
import shutil
import subprocess
import sys

def run(command, cwd=None, env=None):
    print('+', ' '.join(map(str, command)))
    subprocess.run(command, cwd=cwd, env=env, check=True)

if not REGISTRY_PATH.is_file():
    raise FileNotFoundError('Run 01_colab_experiment_setup.ipynb first')

if not (REPO_DIR / '.git').exists():
    run(['git', 'clone', '--branch', REPO_BRANCH, '--depth', '1', REPO_URL, str(REPO_DIR)])
else:
    run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', REPO_BRANCH])
    run(['git', '-C', str(REPO_DIR), 'checkout', REPO_BRANCH])
    run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', REPO_BRANCH])

run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO_DIR / 'requirements.lock')])
run(['bash', 'run.sh', 'data'], cwd=REPO_DIR)
os.environ['PYTHONPATH'] = str(REPO_DIR / 'src')
sys.path.insert(0, str(REPO_DIR / 'src'))

## 3. Build the exact execution plan

`pool_size` and `eval_sizes` differ deliberately. For example, the small preset extracts 50 documents and evaluates 25-document samples so that the five seeds actually select different documents.

In [ ]:
import pandas as pd

registry = json.loads(REGISTRY_PATH.read_text(encoding='utf-8'))
if SIZE_MODE not in registry['size_presets']:
    raise ValueError(f'Unknown SIZE_MODE={SIZE_MODE!r}')
size_spec = registry['size_presets'][SIZE_MODE]
pool_size = int(size_spec['pool_size'])
eval_sizes = list(map(int, size_spec['eval_sizes']))
eval_seeds = list(map(int, size_spec['seeds']))
baseline_prompt = registry['baseline_prompt_name']
baseline_relations = int(registry['baseline_relation_count'])
variants = list(registry['variants'])

if FOCUS == 'prompts':
    selected = [v for v in variants if v['relation_count'] == baseline_relations]
elif FOCUS == 'relations':
    selected = [v for v in variants if v['prompt_name'] == baseline_prompt]
elif FOCUS == 'factorial':
    selected = variants
elif FOCUS == 'baseline':
    selected = [v for v in variants if v['prompt_name'] == baseline_prompt and v['relation_count'] == baseline_relations]
elif FOCUS == 'custom':
    selected = [v for v in variants
                if (not CUSTOM_PROMPTS or v['prompt_name'] in CUSTOM_PROMPTS)
                and (not CUSTOM_RELATION_COUNTS or v['relation_count'] in CUSTOM_RELATION_COUNTS)]
else:
    raise ValueError(f'Unknown FOCUS={FOCUS!r}')

if not selected:
    raise ValueError('Variant filters selected nothing')

plan = pd.DataFrame([{
    'variant_id': v['variant_id'],
    'prompt': v['prompt_name'],
    'relations': v['relation_count'],
    'relation_coverage': v['train_relation_coverage_share'],
    'pool_docs': pool_size,
    'eval_sizes': ','.join(map(str, eval_sizes)),
    'seeds': ','.join(map(str, eval_seeds)),
} for v in selected])
display(plan)
print(f'Variants: {len(selected)}')
print(f'Maximum new LLM document calls: {len(selected) * pool_size}')
print('Existing matching raw caches will be resumed rather than repeated.')

## 4. Ollama

This cell starts the server only when `EXECUTE_RUNS=True`. The model name is read from the selected variant's YAML, preventing the notebook and configuration from accidentally diverging.

In [ ]:
import time
import urllib.request
import yaml

OLLAMA_MODELS = DRIVE_ROOT / 'ollama_models'
OLLAMA_MODELS.mkdir(parents=True, exist_ok=True)

def ollama_ready():
    try:
        urllib.request.urlopen('http://127.0.0.1:11434/api/tags', timeout=2).read()
        return True
    except Exception:
        return False

if EXECUTE_RUNS:
    run(['nvidia-smi'])
    if shutil.which('ollama') is None:
        if shutil.which('zstd') is None:
            run(['apt-get', 'update', '-qq'])
            run(['apt-get', 'install', '-y', 'zstd'])
        ollama_installer = Path('/tmp/install_ollama.sh')
        urllib.request.urlretrieve('https://ollama.com/install.sh', ollama_installer)
        run(['sh', str(ollama_installer)])
    os.environ['OLLAMA_MODELS'] = str(OLLAMA_MODELS)
    os.environ['OLLAMA_HOST'] = '127.0.0.1:11434'
    if not ollama_ready():
        ollama_log = open('/content/ollama.log', 'ab', buffering=0)
        ollama_process = subprocess.Popen(
            ['ollama', 'serve'], stdout=ollama_log, stderr=subprocess.STDOUT, env=os.environ.copy())
        for _ in range(60):
            if ollama_ready():
                break
            time.sleep(1)
        else:
            raise RuntimeError('Ollama did not start; inspect /content/ollama.log')
    first_cfg = yaml.safe_load(Path(selected[0]['config_path']).read_text(encoding='utf-8'))
    model_name = first_cfg['extractor']['model']
    run(['ollama', 'pull', model_name], env=os.environ.copy())
else:
    print('Dry plan only: Ollama was not started.')

## 5. Helper functions

A variant's master raw cache may grow over time. Each run creates an exact subset containing the first `pool_size` documents, so rerunning small after large remains reproducible.

In [ ]:
import csv
import math
from statistics import mean

from kgrepair.config import load_config, rpath
from kgrepair.data import gold_triples, load_split
from kgrepair.evaluate import sample_docs
from kgrepair.extract import load_raw
from kgrepair.pipeline import DocRun, FLAGS_BY_NAME
from kgrepair.schema import load_schema

def create_raw_subset(master_path, subset_path, expected_docs):
    latest = {}
    with open(master_path, encoding='utf-8') as stream:
        for line in stream:
            if not line.strip():
                continue
            record = json.loads(line)
            doc_id = int(record['doc_id'])
            if doc_id < expected_docs:
                latest[doc_id] = record
    missing = set(range(expected_docs)) - set(latest)
    if missing:
        raise ValueError(f'Raw cache is missing {len(missing)} requested documents')
    subset_path.parent.mkdir(parents=True, exist_ok=True)
    with open(subset_path, 'w', encoding='utf-8') as stream:
        for doc_id in range(expected_docs):
            stream.write(json.dumps(latest[doc_id], ensure_ascii=False) + '\n')

def export_trial_kgs(cfg, raw_path, split, out_dir, config_names):
    schema = load_schema(cfg)
    docs = load_split(rpath(cfg, cfg['paths'][split]))
    raw = load_raw(raw_path)
    out_dir.mkdir(parents=True, exist_ok=True)
    counts = {}
    for config_name in config_names:
        if config_name not in FLAGS_BY_NAME:
            raise ValueError(f'Unknown KG configuration {config_name!r}')
        c, s, p = FLAGS_BY_NAME[config_name]
        count = 0
        safe_name = config_name.replace('+', '_plus_')
        with open(out_dir / f'edges_{safe_name}.jsonl', 'w', encoding='utf-8') as stream:
            for doc_id, record in sorted(raw.items()):
                result = DocRun(docs[doc_id], record['parsed'], schema, cfg, judge=None).run(c, s, p)
                for edge in result['edges']:
                    head = result['nodes'].get(edge['head'], {})
                    tail = result['nodes'].get(edge['tail'], {})
                    row = {
                        'doc_id': doc_id, 'title': docs[doc_id]['title'], 'config': config_name,
                        'head_id': edge['head'], 'head': head.get('name', edge['head']),
                        'head_types': sorted(head.get('types', [])),
                        'relation': edge['rel'],
                        'relation_name': schema['relations'].get(edge['rel'], {}).get('name'),
                        'tail_id': edge['tail'], 'tail': tail.get('name', edge['tail']),
                        'tail_types': sorted(tail.get('types', [])),
                        'source_raw_ids': edge['src'],
                    }
                    stream.write(json.dumps(row, ensure_ascii=False) + '\n')
                    count += 1
        counts[config_name] = count
    return counts

def enrich_metrics(auto_metrics_path, cfg, raw_path, full_relation_ids):
    metrics = pd.read_csv(auto_metrics_path)
    docs = load_split(rpath(cfg, cfg['paths'][SPLIT]))
    raw = load_raw(raw_path)
    pool = sorted(raw)
    full_relations = set(full_relation_ids)
    common_gold = {}
    for seed in sorted(metrics['seed'].unique()):
        for n_docs in sorted(metrics['n_docs'].unique()):
            ids = sample_docs(pool, int(n_docs), int(seed))
            common_gold[(int(seed), int(n_docs))] = sum(
                len(gold_triples(docs[doc_id], full_relations)) for doc_id in ids)
    metrics['gold_common_scope'] = [common_gold[(int(r.seed), int(r.n_docs))] for r in metrics.itertuples()]
    metrics['recall_common_scope'] = metrics['tp_unique'] / metrics['gold_common_scope'].replace(0, float('nan'))
    denom = metrics['unique_precision'] + metrics['recall_common_scope']
    metrics['f1_common_scope'] = (
        2 * metrics['unique_precision'] * metrics['recall_common_scope'] / denom.replace(0, float('nan'))
    ).fillna(0.0)
    return metrics


## 6. Run the selected variants

For every variant, the notebook saves the raw cache, runtime configuration, all standard evaluation files, enriched metrics against the shared full relation catalog from the registry, trial KGs, and a run summary.

In [ ]:
from datetime import datetime, timezone

comparison_rows = []

if not EXECUTE_RUNS:
    print('EXECUTE_RUNS=False: no LLM calls or evaluation were performed.')
else:
    for index, variant in enumerate(selected, 1):
        variant_id = variant['variant_id']
        print(f'\n===== [{index}/{len(selected)}] {variant_id} =====')
        run_dir = EXPERIMENT_ROOT / 'runs' / variant_id / SPLIT / SIZE_MODE
        metrics_dir = run_dir / 'metrics'
        run_dir.mkdir(parents=True, exist_ok=True)
        metrics_dir.mkdir(parents=True, exist_ok=True)

        cfg_data = yaml.safe_load(Path(variant['config_path']).read_text(encoding='utf-8'))
        cfg_data['eval']['sizes'] = eval_sizes
        cfg_data['eval']['seeds'] = eval_seeds
        cfg_data['eval']['primary_n'] = max(eval_sizes)
        cfg_data['eval']['primary_seed'] = eval_seeds[0]
        cfg_data['paths']['results_dir'] = str(metrics_dir)
        runtime_config = run_dir / 'config.runtime.yaml'
        runtime_config.write_text(yaml.safe_dump(cfg_data, sort_keys=False, allow_unicode=True), encoding='utf-8')

        master_raw = EXPERIMENT_ROOT / 'raw' / variant_id / f'{SPLIT}.jsonl'
        master_raw.parent.mkdir(parents=True, exist_ok=True)
        extract_command = [
            sys.executable, '-m', 'kgrepair', '--config', str(runtime_config),
            'extract', '--split', SPLIT, '--n', str(pool_size), '--out', str(master_raw),
        ]
        started = time.perf_counter()
        run(extract_command, cwd=REPO_DIR, env=os.environ.copy())
        extraction_wall_s = time.perf_counter() - started

        subset_raw = run_dir / f'raw_pool_{pool_size}.jsonl'
        create_raw_subset(master_raw, subset_raw, pool_size)
        evaluate_command = [
            sys.executable, '-m', 'kgrepair', '--config', str(runtime_config),
            'evaluate', '--raw', str(subset_raw), '--split', SPLIT, '--no-judge',
            '--out-dir', str(metrics_dir),
        ]
        run(evaluate_command, cwd=REPO_DIR, env=os.environ.copy())

        cfg = load_config(runtime_config)
        raw = load_raw(subset_raw)
        normal = sum(r['parsed'] is not None and r.get('parse_error') is None for r in raw.values())
        partial = sum(r.get('parse_error') == 'partial_json_recovered' for r in raw.values())
        failed = sum(r['parsed'] is None for r in raw.values())
        latencies = [float(r['latency']) for r in raw.values() if r.get('latency') is not None]
        completion_tokens = [
            int((r.get('usage') or {}).get('completion_tokens') or 0) for r in raw.values()
        ]

        enriched = enrich_metrics(
            metrics_dir / 'auto_metrics.csv', cfg, subset_raw, registry['full_relation_ids'])
        enriched_path = metrics_dir / 'auto_metrics_enriched.csv'
        enriched.to_csv(enriched_path, index=False)

        kg_counts = {}
        if EXPORT_TRIAL_KG:
            kg_counts = export_trial_kgs(cfg, subset_raw, SPLIT, run_dir / 'kg', KG_CONFIGS)

        run_summary = {
            'created_at_utc': datetime.now(timezone.utc).isoformat(),
            'variant_id': variant_id, 'prompt_name': variant['prompt_name'],
            'relation_count': variant['relation_count'],
            'relation_ids': variant['relation_ids'],
            'train_relation_coverage_share': variant['train_relation_coverage_share'],
            'split': SPLIT, 'size_mode': SIZE_MODE, 'pool_size': pool_size,
            'eval_sizes': eval_sizes, 'seeds': eval_seeds,
            'normal_parses': normal, 'partial_recoveries': partial, 'parse_failures': failed,
            'mean_model_latency_s': mean(latencies) if latencies else None,
            'total_completion_tokens': sum(completion_tokens),
            'current_cell_extraction_wall_s': extraction_wall_s,
            'kg_edge_counts': kg_counts,
            'raw_path': str(subset_raw), 'metrics_dir': str(metrics_dir),
        }
        (run_dir / 'experiment_run.json').write_text(
            json.dumps(run_summary, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')

        for (config_name, n_docs), group in enriched.groupby(['config', 'n_docs']):
            comparison_rows.append({
                'variant_id': variant_id, 'prompt_name': variant['prompt_name'],
                'relation_count': variant['relation_count'],
                'train_relation_coverage_share': variant['train_relation_coverage_share'],
                'split': SPLIT, 'size_mode': SIZE_MODE, 'pool_size': pool_size,
                'config': config_name, 'n_docs': int(n_docs), 'n_seeds': len(group),
                'precision_mean': group['precision'].mean(),
                'unique_precision_mean': group['unique_precision'].mean(),
                'recall_in_scope_mean': group['recall'].mean(),
                'f1_in_scope_mean': group['f1'].mean(),
                'recall_common_scope_mean': group['recall_common_scope'].mean(),
                'f1_common_scope_mean': group['f1_common_scope'].mean(),
                'retention_raw_mean': group['retention_raw'].mean(),
                'correct_removed_share_mean': group['correct_removed_share'].mean(),
                'normal_parses': normal, 'partial_recoveries': partial, 'parse_failures': failed,
                'mean_model_latency_s': mean(latencies) if latencies else None,
                'total_completion_tokens': sum(completion_tokens),
                'run_dir': str(run_dir),
            })

    comparison = pd.DataFrame(comparison_rows)
    comparison_path = EXPERIMENT_ROOT / 'comparisons' / f'{FOCUS}__{SPLIT}__{SIZE_MODE}.csv'
    comparison.to_csv(comparison_path, index=False)
    print('Comparison saved:', comparison_path)

## 7. Review the comparison

`in_scope` evaluates only the relations included in a particular variant. `common_scope` uses the shared gold universe of the complete relation catalog recorded in the registry, preventing a smaller relation set from winning artificially by removing part of the task. When comparing prompts under the same schema, both metric families are directly comparable.

In [ ]:
comparison_path = EXPERIMENT_ROOT / 'comparisons' / f'{FOCUS}__{SPLIT}__{SIZE_MODE}.csv'
if comparison_path.exists():
    comparison = pd.read_csv(comparison_path)
    main_view = comparison[comparison['config'].isin(['raw', 'C+S'])].sort_values(
        ['n_docs', 'config', 'f1_common_scope_mean'], ascending=[True, True, False])
    display(main_view)
else:
    print('No comparison file yet. Set EXECUTE_RUNS=True and run the execution cell.')

## After the experiments

The runner deliberately does not declare an automatic winner. First compare quality, parse failures, latency, token usage, and relation coverage. Then confirm a small number of leading candidates on DEV.

After manually selecting the prompt and relation set, copy them into the final extraction notebook as literal constants. Do not make the final notebook automatically read the 'best' variant from a comparison CSV: the selected configuration must be explicitly fixed before TEST.